In [ ]:
%load_ext autoreload
%autoreload 2

# Aserver AGen extension

- so far, pya.Aserver was only capable of playing Asigs
- AGens so far had to be rendered first, then played via `ag.gen_asig().play()`
- with this Aserver extension, it is possible to dispatch AGens directly.
- Technically, Aserver sorts dispatched items in a list sorted by onset
  - when dispatched time has come, (in case of asigs) blocksize snippets are
    taken from the buffer and merged into output. 
  - for AGens, the logic is different: 
    - in the server, the next blocksize samples are computed (directly), allowing
      - real-time / online synthesis, i.e., continuous rendering
      - real-time modulation of AGens
      - memory-saving (yet computationally probably more expensive) synthesis
  - meanwhile I added a more general ResamplerGen allowing
    - to set an arbitrary rate argument
    - resampling flexibly from any rate to any target rate
- The current implementation has to be improved. TODOs are:
  - Better processing of multi-channel AGens (>2, resp. >output bus). This  needs 
    implementation and checks
  - Cache management needs to be improved, e.g. invalidating stuff older than 2 mins 
  - load-monitoring could be implemented and deactivate agens that would cause
    chopping aserver operation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

import time

# import pyamapping as pam
from pya import startup, device_info
from pya.agen.core import stereo, multi_channel
from pya.agen.lib import Line, SinOsc, WhiteNoise, expand_channels

mpl.rcParams["figure.figsize"] = (9, 3)

In [ ]:
device_info(); # check devices and their IDs

In [ ]:
s = startup() # specify/set device if needed

## First test of Aserver play_agen() / agen extension

#### Test a continuously rendering Agen (i.e. without done='stop')

In [ ]:
# as the old way / reference: play ag1 via the already possible 'Asig detour'...
ag1 = SinOsc(SinOsc(6) * 30 + 300) * 0.25
ag1.gen_asig(seconds=0.5).stereo().play(onset=0.2)

In [ ]:
# now play via `s.play_agen()``  (aka dispatch via AServer)
s.play_agen(ag1, onset=0.2, out=0) # this plays forever (until stopped via the next line)

In [ ]:
s.stop() # stop (delete all dispatched items on Aserver)

AGen.play() is a more convenient interface
- Actually AGen.play() calls Aserver.play_agen()

In [ ]:
ag1.play(onset=0.2)
time.sleep(1)
s.stop()

#### Test a finite Agen (i.e. one that ends after some time)

In [ ]:
ag2 = WhiteNoise() * Line(0.5, 0, 0.5)
ag2.play(onset=0.2)

In [ ]:
s.play_agen(ag2, onset=0)

In [ ]:
ag2.play(onset=0) # now play via the AGen.play(), which calls s.play_agen() to dispatch

#### Test realtime modulation of a playing AGen 

In [ ]:
p_freq, p_vib = [400], [10]
agmod = SinOsc(SinOsc(5) * p_vib + p_freq) * 0.2
s.play_agen(agmod, 0, 0)

In [ ]:
p_freq[0] *= 1.059 # feel free to execute repeatedly

In [ ]:
p_freq[0] /= 1.059 # feel free to execute repeatedly

In [ ]:
p_vib[0] *= 2  # feel free to execute repeatedly

In [ ]:
# interactive UI-based parameter adjustments
from ipywidgets import interactive
def syngui(freq=200, vib=2):
    global p_freq, p_vib
    p_freq[0], p_vib[0] = freq, vib
interactive(syngui, freq=(100, 700, 1), vib=(0, 20, 0.2))

In [ ]:
s.stop()

#### Test multichannel (resp. stereo) AGens

In [ ]:
ag2ch = SinOsc(freq=stereo(600, 800)) * expand_channels(Line(1, 0, 4.2, curve=-4), 2, 'last')

In [ ]:
ag2ch = SinOsc(freq=stereo(600, 800)) * Line(1, 0, 4.2, curve=-4)

In [ ]:
ag2ch.gen_asig().play(onset=0.5).plot(offset=2, lw=0.2) # the old way

In [ ]:
ag2ch.play() # the new (online rendering) way: hurray that works

## MouseX, MouseY - AGen sensors for real-time rendering

Now that we can render in realtime, let's create helper sensors as AGen to
modulate AGen nodes interactively. 

- For getting the Mouse position, `pyautogui` seems applicable. 
- ToDo: check whether there is a more light-weight solution

In [ ]:
import pyautogui

# Get the current mouse position
current_mouse_position = pyautogui.position()

# Print the mouse position
print(f"Current mouse position: {current_mouse_position}")

Here are two custom AGens as Realtime Sensors. 
- maybe they can be added to agen.lib later, but first they need to mature

In [ ]:
from pya.agen.core import SingleChannelGen

class MouseX(SingleChannelGen):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        import pyautogui

    def _generate_single(
        self, 
        sample_count: int, # The amount of samples that should be generated
        start: int, # The index of the first sample
    ) -> np.ndarray:
        # block_num = self.state.data.get("block_num", 0)
        # self.state.data["block_num"] = block_num + 1
        current_mouse_position = pyautogui.position()
        return np.full(sample_count, current_mouse_position.x)
    
class MouseY(SingleChannelGen):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        import pyautogui

    def _generate_single(
        self, 
        sample_count: int, # The amount of samples that should be generated
        start: int, # The index of the first sample
    ) -> np.ndarray:
        # block_num = self.state.data.get("block_num", 0)
        # self.state.data["block_num"] = block_num + 1
        current_mouse_position = pyautogui.position()
        return np.full(sample_count, current_mouse_position.y)

In [ ]:
# MouseX test: execute, then move mMuse pointer horizontally
for i in range(30):
    print("move Mouse pointer! ==> Mouse.x = ", MouseX(sr=10).gen_asig(1).sig, end='\r')
    time.sleep(0.1)

Now a first online synthesis using MouseX and MouseY

In [ ]:
agfreq = MouseX().linlin(0, 2560, 200, 400) # x for frequency
agvib = MouseY().linlin(0, 2000, 0, 20) # y for vibrato speed
(SinOsc(agfreq) * SinOsc(agvib).linlin(-1,1,0,0.5)).play();

In [ ]:
s.stop()

**Benchmarking:** 
- MouseX fills all values with current pointer.x coordinate. 
- this is executed only once per block: i.e. 
- with current Aserver defaults (s=44100, bs=512) at ~86 Hz
- Here the timing of a single call (i.e. position queried once per generate)
  - its in the order of 0.5 ms on my M2 MBP

In [ ]:
t = time.time(); MouseX().generate(512, 0, 0); print(f"used time = {(time.time() -t)* 1000:4.2f} milliseconds")

In [ ]:
# freq_agens = stereo(MouseX(sr=50), MouseY(sr=25))
# (SinOsc.ar(freq_agens) * Line(1,0,0.5, sr=42)).play(rate=1.5)

**ToDos**: das soll das Richtige macht, d.h.: 

-  es geht wenn Aserver mit 48000 Hz läuft, agen aber mit 44100 rendered
-  es ist definierbar, mit welcher frame rate das Mouse scanning läuft) 
-  es wird beim upsampling automatisch linear interpoliert zwischen MouseX values
-  ein implizites Channel broadcasting vereinfacht es, n-Kanal und 1-Kanal Audio via operator (hier "*" zu verknüpfen (Details zu klären, ginge über default "last")
-  via rate kann man angeben und dann werden weniger (bzw. mehr) samples/s computed und dazwischen auch korrekt interpoliert.
- das neue resampling könnte nebenbei auch das upsampling-Problem lösen (once and for all ;-) 


In [ ]:
freq_agens = stereo(MouseX(sr=50), MouseY(sr=25))

In [ ]:
SinOsc(freq_agens).gen_asig(seconds=0.02).plot()

In [ ]:
(SinOsc.ar(200) * Line.kr(1,0,0.5)).play(rate=1)

In [ ]:
freq_agens = stereo(MouseX(sr=22050), MouseY(sr=22050))
(SinOsc.ar(freq_agens) * Line(1, 0, 3.5, sr=44100)).play(rate=1.5)

In [ ]:
%matplotlib inline
# %matplotlib widget 
plt.close("all")

## New ResampleGen
- Goal: resample to any target sampling rate with additional rate scaling

First test of the new ResampleGen
- A SinOsc at nyquist frequency gives a [1,-1,1,-1,...] signal (at correct phase)
- The ResampleGen should upsample this to audio delivering a Tri shaped signal
- The rate = 0.5 should double the interpolated signal's duration to 200% (red dots)

In [ ]:
from pya.agen.core import ResampleGen
from pya.agen.lib import SinOsc, Line

ag1 = SinOsc(500, phase=np.pi/2, sr=1000) * Line(1, 0, 0.1, sr=1000)
plt.figure()
ag1.gen_asig().plot(marker=".", lw=0.1)

rag = ResampleGen(ag1, rate=0.5, sr=44100)
rag.gen_asig().plot(marker=",", color="r", ms=1.5, lw=0.2)
rag.play(onset=0);

- investigate stability of resampling, varying various arguments
- blocksize bs doesn't show visible changes (which is what is wanted...)

In [ ]:
from ipywidgets import interactive

# %matplotlib inline
%matplotlib widget 
plt.close("all")
plt.figure()
agen_sr = 50
ag = SinOsc(25, sr=agen_sr, phase=np.pi / 2) * Line(1, 0, 1, sr=agen_sr)
ag.to_sr(540, 0.3).gen_asig(block_size=100).plot(marker=".", ms=1.2, lw=0.1)

def resample_test(agen_sr=50, freq=25, target_sr=400, rate=1.0, bs=100):
    plt.clf()
    ag = SinOsc(freq, sr=agen_sr, phase=np.pi / 2) * Line(1, 0, 1, sr=agen_sr)
    ag.to_sr(target_sr, rate).gen_asig(block_size=bs).plot(marker=".", ms=1.2, lw=0.1)

interactive(resample_test, agen_sr=(1, 100, 1), target_sr=(10, 4000, 1), rate=(0.1, 10, 0.1), bs=(10, 1000, 1))

In [ ]:
(SinOsc(300)*0.2*(ag.to_sr(22050, rate=0.3))).play()

In [ ]:
%matplotlib inline 
plt.close("all")

In [ ]:
(SinOsc(300)*0.2*(ag | ResampleGen.p(0.3, sr=44100))).gen_asig().plot(lw=0.1)

Test resampling with recorded sound (here a mono finger snap)

In [ ]:
from pya import Asig
from pya.agen.lib import PlayAsig, Pan2

a1 = Asig("samples/snap.wav")[{0.027:None}]  # load finger snap to asig
a1.plot()  # plot

a1g = PlayAsig(a1) # create an AGen

# offline render (the old way, with rate=0.2)
ResampleGen(a1g, rate=0.2).gen_asig().plot(lw=1, ls="-.", color="r")

# demonstrate implicit to_sr() via rate parameter
a1g.play(onset=0, rate=1)
a1g.play(onset=1, rate=0.25, channel=1)
a1g.play(onset=2, rate=0.05, channel=0)

# demonstrate spatial panning 
(ResampleGen(a1g, 0.1) | Pan2.p(0)).play(onset=4);

Test ResampleGen with stereo AGens
- first a stereo wav file
- then some spatialized impulses

In [ ]:
ast = Asig("samples/stereoTest.wav") # .play()
# PlayAsig(ast).gen_asig().play(rate=2)
PlayAsig(ast).play(rate=1)

In [ ]:
from pya.agen.lib import BLImp, Pan2
agtic = (BLImp(3, 2000) | Pan2.p(SinOsc(1.5, phase=-np.pi/2))) * Line(1,1,2.1)
agtic.gen_asig().plot(offset=2) 
agtic.to_sr(rate=0.5).play();

**====================================
 (end of examples - following next:) 
====================================**

## Developing notes - Experiments with chunkwise computation of AGens

before modifying Aserver, let's first check block-wise generation, as it should later be done on dispatched AGens.

In [ ]:
plt.close("all")
%matplotlib inline

In [ ]:
idx = 0

In [ ]:
# check howto compute consequtive sample blocks
# execute repeatedly to see phase continuation
if idx == 0:
    ag1 = SinOsc(SinOsc(100) * 1000 + 1500)
    ag1.create_graph()
bs = 256
d = ag1.generate(bs, idx, 0)
idx += bs
plt.plot(d);

Disclaimer: 
- The above will only work if the AGens sample rate matches the 
  Aserver's samplerate and if play rate=1
- I checked the previous ResampleGen (now commented out / replace) but this is 
- quite limited as it only supports situations where one by the other is an integer.
- Find below a better solution for more flexible resampling. 
  - I think it is a candidate for deeper pya.agen integration 
  - it could (maybe) also solve the upsampling problem mentioned in our AMICAD 2025 paper. 

To create a solution that works generally let's use interp (like in Asig.resample()

- I want the rate argument r to effectively play sr/r of the input samples per second
- yet at the moment I implicitly assume that the sampling rate of the agen matches the one of the aserver.
- if this is not the case, it would require a resample anyway.
- Assume we have
  - Aserver sampling rate srs 
  - AGen sampling rate sra, 
  - .play() rate argument r
  - blocksize of server bs

In [ ]:
# test AGen to work with:
ag1 = SinOsc(90, phase=0.5, sr=1000) * Line(1, 0, 0.1, sr=1000)

# parameters
gen = ag1 
server_sr = s.sr
bs = s.bs
agen_sr = gen.sr
rate = 1.5

# initialize generation (to be done on dispatching)
ch = 0
gen_pos = 0
agen_latest_idx = 0
agen_sample_incr = agen_sr / server_sr * rate
agen_block_increment = agen_sample_incr * bs
agen_needed_samples = int(agen_block_increment + 1)

result = []

execute the cell below a couple of times and see how rendered samples (blue dots) 
are a correct linear interpolation between the input samples (red dots)

In [ ]:
# on each generation cycle for blocksize samples do this:
agen_pos_end = gen_pos + bs * agen_sample_incr

# for all channels (add loop over ch here)
gen.generate(agen_needed_samples+1, agen_latest_idx, channel=ch)

# take values for interpolation from cache
agen_sig_val = gen.states[ch].get_from_cache(sample_count=agen_needed_samples+1, start=int(gen_pos))
npoints = agen_sig_val.shape[0]
agen_sig_pos = np.arange(int(gen_pos), int(gen_pos) + npoints)

if npoints > 0: # if there is still agen data
    dest_pos = np.arange(gen_pos, gen_pos + agen_block_increment, agen_sample_incr)
    dest_sig = np.interp(dest_pos, agen_sig_pos, agen_sig_val)
    result.append(dest_sig)    
    # now we have our result: add dest_sig to output
    plt.plot(agen_sig_pos, agen_sig_val, "r.")
    plt.plot(dest_pos, dest_sig, "b,")
    plt.ylim(-1, 1)

    agen_latest_idx += agen_needed_samples
    gen_pos += agen_block_increment
else: 
    pass # end reached

In [ ]:
# plot the full data (appended above only for inspection)
plt.plot(np.concatenate(result), ".", ms=1);

**Meanwhile done:**: 
- integrate the above-sketched interpolation strategy (-> ResampleGen now in agen.core)
- ToDo: consider to make this the default strategy to deal with up-/downsampling when
  using differing sample rates in AGens.
- ToDo: consider to implement with rate as GenOrNum

In [ ]:
# # this was the first working version of a general Resampler -> improved below
# from pya.agen.core import AGen, AGenState
# from pya.agen.core import SingleChannelGen

# class Resample(SingleChannelGen):
#     def __init__(self, agen: AGen, rate: float = 1.0, *args, **kwargs):
#         super().__init__(*args, **kwargs)
#         self.state = AGenState()
#         self.in_sample_incr = agen.sr / self.sr * rate
#         self.agen = agen
#         self.rate = rate

#     def _generate_single(
#         self,
#         sample_count: int,  # The amount of samples that should be generated
#         start: int,  # The index of the first sample
#     ) -> np.ndarray:
#         in_pos = self.state.data.get("in_pos", 0)
#         in_idx = self.state.data.get("in_idx", 0)

#         self.in_block_increment = self.in_sample_incr * sample_count
#         max_in_pos = in_pos + self.in_block_increment

#         # plt.axvline(in_pos / self.in_sample_incr / self.sr, color="r", lw=0.3)

#         # since we have data until in_idx we need to generate from in_idx to ceil(max_in_pos)
#         n_render = int(max_in_pos + 1) - in_idx + 1

#         # ToDo: check with Luka what the best way is to initiate rendering
#         ch = 0  # for all channels (add loop over ch here)
#         agen_ended_flag = False
#         nr_generated = len(self.agen.generate(n_render, in_idx, channel=ch))
#         in_new_idx = in_idx + nr_generated
#         if nr_generated < n_render:  # i.e. agen ended at in_new_idx
#             agen_ended_flag = True  # then in_new_idx is the max allowed index
#         self.state.data["in_idx"] = in_new_idx

#         # now let's get the agen samples for interpolation
#         if in_pos == 0:
#             start = 0
#             in_sig_val = self.agen.states[ch].get_from_cache(
#                 sample_count=n_render, start=start
#             )
#         else:
#             start = max(int(in_pos) - 2, 0)
#             in_sig_val = self.agen.states[ch].get_from_cache(
#                 sample_count=in_new_idx - start, start=start
#             )
#         n_points = in_sig_val.shape[0]
#         in_sig_pos = np.linspace(start, start + n_points, n_points, endpoint=False)

#         if n_points > 0:
#             n = sample_count
#             dest_pos = np.linspace(in_pos, max_in_pos, n, endpoint=False)
#             dest_sig = np.interp(dest_pos, in_sig_pos, in_sig_val)
#             if agen_ended_flag:  # truncate extra samples
#                 endidx = np.searchsorted(dest_pos, in_new_idx, side="left")
#                 dest_sig = dest_sig[:endidx]
#             self.state.data["in_pos"] = in_pos + self.in_block_increment
#             return dest_sig
#         else:
#             return np.empty(0)

# def to_sr_fn(self, sr=44100, rate=1):
#     return Resample(self, rate=rate, sr=sr)

# AGen.to_sr = to_sr_fn

In [ ]:
# optimized code for the above initial implementation
# Added to pya.agen.lib - on 2025-07-26
# from pya.agen.core import AGen, AGenState
# from pya.agen.core import SingleChannelGen
# class Resample(SingleChannelGen):
#     def __init__(self, gen: AGen, rate: float = 1.0, *args, **kwargs):
#         super().__init__(*args, **kwargs)
#         self.state = AGenState()
#         self.sample_incr = gen.sr / self.sr * rate
#         self.agen = gen
#         self.rate = rate

#     def _generate_single(
#         self,
#         sample_count: int,  # The amount of samples that should be generated
#         start: int,  # The index of the first sample
#     ) -> np.ndarray:
#         gen_pos = self.state.data.get("gen_pos", 0)
#         start_idx = max(0, int(gen_pos)-1)
#         gen_stop_pos = gen_pos + self.sample_incr * sample_count
#         n_render = int(gen_stop_pos + 1) - start_idx + 1

#         ch = 0  # TODO: for all channels (add loop over ch here)?
#         src_sig = self.agen.generate(n_render, start_idx, channel=ch)
#         n_pts = src_sig.shape[0]
#         src_pos = np.arange(0, n_pts) + start_idx # faster than np.linspace

#         if n_pts < n_render: # limit output if input end was reached 
#             stop_index = start_idx + n_pts
#             m = int((min(stop_index, gen_stop_pos) - gen_pos) / self.sample_incr)
#             dest_max_pos = gen_pos + m * self.sample_incr
#         else:
#             m = sample_count
#             dest_max_pos = gen_stop_pos
#         dest_pos = np.linspace(gen_pos, dest_max_pos, m, endpoint=False)
#         dest_sig = np.interp(dest_pos, src_pos, src_sig)

#         self.state.data["gen_pos"] = gen_pos + m * self.sample_incr

#         if n_pts > 0:
#             return dest_sig
#         else:
#             return np.empty(0)